## Fremont Bridge Cycling Dashboard
### Visualisation Techniques 
**Author:** Arkadiusz Jedrzejewski 
**Date:** June 2026

In [1]:
# Install required packages
import sys
!{sys.executable} -m pip install dash plotly # APPENDING THIS LINE TO INSTALL DASH AND PLOTLY

# Create a Dash application
from dash import Dash, dcc, html, Input, Output
import pandas as pd
import numpy as np
import plotly.express as px

In [3]:
# Load the dataset
df = pd.read_csv('../DATASETS/Fremont_Bridge_-_Combined_Bicycle_and_Scooter_Counter_20260615.csv')

In [4]:
# Display the first 5 rows of the dataset
df.head(5)

,Date,"Fremont Bridge Sidewalks, south of N 34th St Total","Fremont Bridge Sidewalks, south of N 34th St Cyclist West Sidewalk","Fremont Bridge Sidewalks, south of N 34th St Cyclist East Sidewalk"
0,05/31/2026 11:00:00 PM,33,9.0,24.0
1,05/31/2026 10:00:00 PM,51,20.0,31.0
2,05/31/2026 09:00:00 PM,64,30.0,34.0
3,05/31/2026 08:00:00 PM,148,72.0,76.0
4,05/31/2026 07:00:00 PM,184,85.0,99.0


In [5]:
# Rename columns for easier access
df.columns = ['Date', 'Total', 'West_Sidewalk', 'East_Sidewalk']
df.head(5)

,Date,Total,West_Sidewalk,East_Sidewalk
0,05/31/2026 11:00:00 PM,33,9.0,24.0
1,05/31/2026 10:00:00 PM,51,20.0,31.0
2,05/31/2026 09:00:00 PM,64,30.0,34.0
3,05/31/2026 08:00:00 PM,148,72.0,76.0
4,05/31/2026 07:00:00 PM,184,85.0,99.0


In [6]:
# Basic dataset information
df.shape
df.info()
df.describe() 
df.isnull().sum()  # Check for missing values  


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119747 entries, 0 to 119746
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           119747 non-null  object 
 1   Total          119717 non-null  object 
 2   West_Sidewalk  119717 non-null  float64
 3   East_Sidewalk  119717 non-null  float64
dtypes: float64(2), object(2)
memory usage: 3.7+ MB


Date              0
Total            30
West_Sidewalk    30
East_Sidewalk    30
dtype: int64

In [8]:
# Convert 'Date' to datetime and 'Total' to numeric
df['Date'] = pd.to_datetime(df['Date'])

# Remove rows with missing values and duplicates
df['Total'] = df['Total'].astype(str)
df['Total'] = df['Total'].str.replace(',', '')
df['Total'] = pd.to_numeric(df['Total'], errors='coerce')

# Remove rows with missing values and duplicates
df.dropna(inplace=True)          
df.drop_duplicates(inplace=True)


# Total is already numeric, no conversion needed
df['Total'] = df['Total'].astype(int)

df.info()



<class 'pandas.core.frame.DataFrame'>
Index: 119717 entries, 0 to 119746
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   Date           119717 non-null  datetime64[ns]
 1   Total          119717 non-null  int64         
 2   West_Sidewalk  119717 non-null  float64       
 3   East_Sidewalk  119717 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1)
memory usage: 4.6 MB


In [9]:
# Convert 'West_Sidewalk' and 'East_Sidewalk' to numeric
df['West_Sidewalk'] = df['West_Sidewalk'].astype(int)
df['East_Sidewalk'] = df['East_Sidewalk'].astype(int)

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 119717 entries, 0 to 119746
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   Date           119717 non-null  datetime64[ns]
 1   Total          119717 non-null  int64         
 2   West_Sidewalk  119717 non-null  int64         
 3   East_Sidewalk  119717 non-null  int64         
dtypes: datetime64[ns](1), int64(3)
memory usage: 4.6 MB


In [10]:
# extracting time features from Date column
df['Hour']      = df['Date'].dt.hour
df['DayOfWeek'] = df['Date'].dt.day_name()
df['Month']     = df['Date'].dt.month
df['Year']      = df['Date'].dt.year

df.head(5)

,Date,Total,West_Sidewalk,East_Sidewalk,Hour,DayOfWeek,Month,Year
0,2026-05-31 23:00:00,33,9,24,23,Sunday,5,2026
1,2026-05-31 22:00:00,51,20,31,22,Sunday,5,2026
2,2026-05-31 21:00:00,64,30,34,21,Sunday,5,2026
3,2026-05-31 20:00:00,148,72,76,20,Sunday,5,2026
4,2026-05-31 19:00:00,184,85,99,19,Sunday,5,2026


In [27]:

# Create a Dash application
app = Dash()

app.layout = html.Div([
    html.H4("Fremont Bridge..."),
    dcc.Graph(id="graph"),
    dcc.RangeSlider(         # ← ten sam suwak!
        id="hour-slider",
        min=0, max=23,
        value=[6, 20],
    ),
])

# Define a callback to update the graph based on the selected hour range
@app.callback(
    Output("graph", "figure"),
    Input("hour-slider", "value"),
)
def update_chart(hour_range):
    low, high = hour_range
    mask = (df['Hour'] >= low) & (df['Hour'] <= high)
    fig = px.line(df[mask], x='Date', y='Total')
    return fig

if __name__ == "__main__":
    app.run(debug=True)

In [11]:
# initialise the app
app = Dash()

# Add controls to build the interaction
app.layout = html.Div([
    html.H4("Fremont Bridge Cycling Dashboard"),
    dcc.Graph(id="graph"),

    html.Label("Select direction:"),
    dcc.RadioItems(
        id="direction",
        options=['Total', 'West_Sidewalk', 'East_Sidewalk'],
        value='Total',
        inline=True
    ),

    html.Label("Select hours:"),
    dcc.RangeSlider(
        id="hour-slider",
        min=0, max=23, step=1,
        marks={0:"0:00", 6:"6:00", 12:"12:00", 18:"18:00", 23:"23:00"},
        value=[6, 20],
        tooltip={"always_visible": True, "placement": "bottom"}
    ),
])

@app.callback(
    Output("graph", "figure"),
    Input("hour-slider", "value"),
    Input("direction", "value")
)
def update_chart(hour_range, direction):
    low, high = hour_range
    mask = (df['Hour'] >= low) & (df['Hour'] <= high)
    fig = px.line(df[mask], x='Date', y=direction,
                  title=f'Cyclists per Hour - {direction}')
    return fig

In [26]:
app.run(debug=True, jupyter_mode='inline')

AssertionError: The setup method 'errorhandler' can no longer be called on the application. It has already handled its first request, any changes will not be applied consistently.
Make sure all imports, decorators, functions, etc. needed to set up the application are done before running it.